In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Dim reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap # <== NOT a scikit-learn library, but follows the interface, installed using umap-learn name

import matplotlib.pyplot as plt
import seaborn as sns
import hvplot.pandas

# Unsupervised models in scikit-learn: clustering and dimentionality reduction

An earlier lecturer presented the scikit-learn library and its overall design. In learning that design, the students were introduced to supervised learning. Like the earlier lecture, the goal of this lecture is not to teach students all details of clustering techniques. This lecture remains an introduction to the API used for relevant tasks in data science.

## Cluster trades into common categories
Our task is to build a table containing common attributes of all US stocks: volume, closing price, number of transactions, average size of trades, etc. Note that "average/mean size of trades" needs to be calculated from a large file containing all trades done for a day. 

Once we have enrichsed end-of-day data for each stock with mean_size calculation, we will start the process of building clusters.

Note that in a real-world scenario, _lots_ more features could be added as input to the clustering algorithm.

## Feature Engineering
Very similar to an earlier lecture

### Read EOD data

In [ ]:
ohlcv_sept_df = pd.read_csv('../../datasets/market_data/ohlcv_2025-sept.csv.zip', parse_dates=["date"])\
                    .sort_values(["ticker", "date"])
ohlcv_sept_df

### Create features per symbol: volatility, total_return, etc.

In [ ]:
ohlcv_sept_df["ret"]        = ohlcv_sept_df.groupby("ticker")["close"].pct_change()     # daily return
ohlcv_sept_df["dollar_vol"] = ohlcv_sept_df["close"] * ohlcv_sept_df["volume"]          # $ traded that day
ohlcv_sept_df["range_pct"]  = (ohlcv_sept_df["high"] - ohlcv_sept_df["low"]) / ohlcv_sept_df["close"]         # intraday range

ohlcv_sept_df

### Calculate market return (needed to calculate "Beta" or correlation w/ market)

In [ ]:
beta_s = ohlcv_sept_df.groupby("date")["ret"].mean().rename("mkt")
beta_s

### Add beta to the dataframe

In [ ]:
ohlcv_sept_df = ohlcv_sept_df.join(beta_s, on='date')
ohlcv_sept_df

### Calculate aggregate features per symbol

In [ ]:
ohlcv_groupby_df = ohlcv_sept_df.groupby('ticker')

In [ ]:
features_df = pd.DataFrame({
    "n_days":         ohlcv_groupby_df.size(),
    "vol":            ohlcv_groupby_df["ret"].std(),                             # daily volatility
    "total_ret":      ohlcv_groupby_df["close"].last() / ohlcv_groupby_df["close"].first() - 1, # month return (momentum)
    "avg_range":      ohlcv_groupby_df["range_pct"].mean(),                      # how wild intraday
    "avg_dollar_vol": ohlcv_groupby_df["dollar_vol"].mean(),                     # liquidity
    "last_price":     ohlcv_groupby_df["close"].last(),                          # price level
    "beta":           ohlcv_groupby_df[["ret", "mkt"]]
                        .apply(lambda x: x["ret"].cov(x["mkt"]) / x["mkt"].var()),
})

# Tickers with <2 usable return days make cov() divide by zero -> NaN beta.

features_df

### Read sectors

In [ ]:
sectors_df = pd.read_csv('../../datasets/market_data/sectors.csv.zip')\
            .rename(columns={"company name": "company", "market cap": "market_cap"})\
             [["ticker", "company", "sector", "market_cap"]]\
             .dropna(subset=["sector"])
sectors_df

### Combine features table with sectors

In [ ]:
features_df = features_df.join(sectors_df.set_index("ticker")[["company", "sector", "market_cap"]], how="left")
features_df

### Drop rows with missing values

In [ ]:
features_df = features_df[features_df.n_days >= 15].dropna(
    subset=["vol", "beta", "total_ret", "avg_range", "avg_dollar_vol", "last_price", "market_cap"])


In [ ]:
features_df

## Build a clustering model

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


### Build a naive clustering model (same as an earlier lecture)

In [ ]:
%%time
RAW_FEATURES = ["vol", "total_ret", "avg_range", "avg_dollar_vol",
                "last_price", "market_cap", "beta"]

naive_model = Pipeline([
    ("scale",  StandardScaler()),
    ("kmeans", KMeans(n_clusters=5, random_state=42, n_init="auto")),
])
features_df["naive_cluster"] = naive_model.fit_predict(features_df[RAW_FEATURES])

Notice the method `fit_predict` is a combination of `fit` and `predict` methods we saw in the supervised learning lecture. We are telling KMeans to _learn the distribution_ and assign the same data to the learned clusters as well.

In [ ]:
naive_model.named_steps['kmeans']

In [ ]:
distances = naive_model.fit_transform(features_df[RAW_FEATURES])
pd.DataFrame(distances, columns=[f"dist_to_cluster_{i}" for i in range(naive_model.named_steps['kmeans'].n_clusters)],
             index=features_df.index).head(3).round(2)

Notice that this time we used the method `fit_transform`. Now we are asking kmeans to learn the distributions using the `fit` function and to `transform` the same data into distances

In [ ]:
features_df.sample(10)

### What does the performance look like?

#### The clusters are extremely sekewed!

In [ ]:
features_df["naive_cluster"].value_counts()

Clusters are fairly skewed. _You should sense trouble, this is NOT good news_

#### The silhouette score is TOO perfect!

In [ ]:
from sklearn.metrics import silhouette_score

In [ ]:
zscores = StandardScaler().fit_transform(features_df[RAW_FEATURES])  # the space KMeans saw
pd.DataFrame(zscores, columns = RAW_FEATURES)

Silhouette score

In [ ]:
silhouette_score(zscores, features_df['naive_cluster'])

Silhouette score is a very common measure of the quality of a cluster (see Wikipedia article for mathematical definition: https://en.wikipedia.org/wiki/Silhouette_(clustering)).

This measure indicates **how close a point is to its own cluster vs other clusters**. This is sometimes called the cohesion/separation score. The scores range between -1 and 1. This measure breaks down when almost all the points are in the same cluster (as is the case in our model)

#### PCA further shows that the data is not evernly distributed
**NOTE** We are using PCA only to show the variance of data across two of the most consequential dimensions. If you wanted to visualize clusters in 2 dimensional space, you would use TSNE (built into Scikit-learn) or UMAP (more modern). They don't work for our purpose because they discard global informaiton in favor of local information. 

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca_raw = PCA(n_components=2).fit_transform(zscores)
pd.concat([
    pd.DataFrame(pca_raw, columns=['x', 'y']).reset_index(), 
    features_df.naive_cluster.reset_index()
    ], axis=1).hvplot.scatter('x', 'y', c='naive_cluster', cmap='viridis')

Reminder that almost all of the points are in cluster zero

In [ ]:
features_df.naive_cluster.value_counts()

### Why is the performance so bad?

In [ ]:
from pandas.plotting import scatter_matrix

In [ ]:
scatter_matrix(features_df[RAW_FEATURES], alpha=0.2, figsize=(10, 10));

In [ ]:
scatter_matrix(np.log(features_df[RAW_FEATURES]+1), alpha=0.2, figsize=(10, 10));

#### Did you notice? In the source table, _everything_ iis super skewed!

In [ ]:
features_df['avg_dollar_vol'].plot.hist(bins=100)

In [ ]:
np.log(features_df['avg_dollar_vol'] + 1).plot.hist(bins=100)

#### The values have completely different orders of magnitude

In [ ]:
features_df[RAW_FEATURES].describe().transpose()[['min', 'max']]

#### There is auto-correlation

In [ ]:
features_df[RAW_FEATURES].corr().style.background_gradient(cmap='coolwarm').format(precision=2)

Notice that `beta` and `vol` are very similar. `avg_dollar_vol` and `market_cap` have unexpectedly high siimilarity

### Let's resolve these issues

In [ ]:
def cluster_and_viz(df, cluster_size):
    # Create a copy so the label column can be added
    draw_df = df.copy()

    # Cluster the dataframe
    draw_df['cluster'] = KMeans(n_clusters=cluster_size, random_state=42, n_init="auto").fit_predict(df)
    #draw_df['cluster'] = draw_df['cluster'].astype('category')
    silhouette_number = silhouette_score(df, draw_df.cluster)

    # Normalize dataframe to feed into PCA
    zscores = StandardScaler().fit_transform(df)  # the space KMeans saw

    # Reduce dimensions via PCA
    pca_raw = PCA(n_components=2).fit_transform(zscores)

    # Convert PCA to dataframe
    pca_df = pd.DataFrame(pca_raw, columns=['x', 'y'], index=df.index)

    # Combine PCA with labels
    combined_df = pd.concat([pca_df, draw_df.cluster], axis=1)
    
    return silhouette_number, draw_df.cluster.value_counts(), combined_df.hvplot.scatter('x', 'y', c='cluster', cmap='viridis')

#### Clean the data and remove outliers
Although...sometimes outliers are the points you want to track!

In [ ]:
features_df.shape

In [ ]:
clean_df = features_df[
    (features_df.last_price >= 1) &         # No silly stocks (back in the day these would be Bulletin Board and Pink Sheet stocks)
    (features_df.avg_dollar_vol >= 1e5) &   # Not super thinly tarded stocks
    (features_df.vol < 0.5) &              # Not to crazy volatility
    (features_df.market_cap > 0)           # Sanity check
    ]

clean_df.shape

In [ ]:
silhouette_number, clusters_count, chart = cluster_and_viz(clean_df[RAW_FEATURES], 5)
silhouette_number

In [ ]:
clusters_count

In [ ]:
chart

#### Let's log transform and scale relevant columns

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

In [ ]:
LOG_FEATURES = ["avg_dollar_vol", "last_price"]      # multiplicative -> log
LIN_FEATURES = ["vol", "total_ret", "beta"]          # already tame -> scale only
MODEL_FEATURES = LOG_FEATURES + LIN_FEATURES


In [ ]:
transformed = ColumnTransformer([
    ("log", Pipeline([("log", FunctionTransformer(np.log10)),
                      ("scale", StandardScaler())]), LOG_FEATURES),
    ("lin", StandardScaler(), LIN_FEATURES),
]).fit_transform(clean_df[MODEL_FEATURES])

# Unfortunately ColumnTransformer returns a numpy array, not a DataFrae (strange, I know)
transformed_df = pd.DataFrame(transformed, columns = MODEL_FEATURES)

In [ ]:
silhouette_number, clusters_count, chart = cluster_and_viz(transformed_df, 5)
silhouette_number

In [ ]:
clusters_count

In [ ]:
chart

The distirbution of values across clusters is MUCH more reasonable, even if the Silhouette score has dropped (the "great" score form earlier was not a true reflection of the quality of clusters anyway).

#### Now let's pick the _number_ of clusters correctly
The number '5' was just an arbitrary choice

In [ ]:
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

In [ ]:
%%time
kmeans_params = list()
for k in range(2,10):
    km = KMeans(n_clusters=k, random_state=42, n_init="auto").fit(transformed_df)
    kmeans_params.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette_score(transformed_df, km.labels_),
        #"calinski_harabasz": calinski_harabasz_score(transformed_df, km.labels_),
        #"davies_bouldin": davies_bouldin_score(transformed_df, km.labels_),
    })

In [ ]:
kmeans_scores = pd.DataFrame(kmeans_params)#.set_index("k")
kmeans_scores

In [ ]:
kmeans_scores.hvplot.line('k', 'inertia')

In [ ]:
kmeans_scores.hvplot.line('k', 'silhouette')

We saw **silhouette score** earlier. Now we also see the **inertia**. Inertia is the sum of squared distances of each point with respect to the center of its cluster. Inertia is most often used as part of the **elbow method.** This method is a heuristic where an "elbow" or a "kin" in the line chart is used to find the idea number of clusters in a k-means cluster.

See https://scikit-learn.org/stable/modules/clustering.html for more details

We had picked the cluster size as 5; 3 would also be reasonable.

### What do these clusters mean?

In [ ]:
transformed_df.shape

In [ ]:
clusters5 = KMeans(n_clusters=5, random_state=42, n_init="auto").fit_predict(transformed_df)

In [ ]:
clusters3 = KMeans(n_clusters=3, random_state=42, n_init="auto").fit_predict(transformed_df)

In [ ]:
transformed_df['clusters_3'] = clusters3
transformed_df['clusters_5'] = clusters5

Put the symbols back in

In [ ]:
transformed_df['symbol'] = clean_df.index

In [ ]:
transformed_df.head()

The following transformations are only for the sns library

In [ ]:
transformed_long_5_df = transformed_df.melt(id_vars='clusters_5', value_vars=MODEL_FEATURES,
                           var_name='feature', value_name='value')
transformed_long_5_df

In [ ]:
# I'm sure there are less ugly ways of doing this - postponed until a future version of this lecture
transformed_long_3_df = transformed_df.melt(id_vars='clusters_3', value_vars=MODEL_FEATURES,
                           var_name='feature', value_name='value')

#### Draw box plots and interpret with your brain

In [ ]:
import seaborn as sns

In [ ]:
sns.catplot(data=transformed_long_5_df, x='clusters_5', y='value', col='feature', col_wrap=5,
            kind='box', height=2.8, aspect=0.75, showfliers=False)

In [ ]:
sns.catplot(data=transformed_long_3_df, x='clusters_3', y='value', col='feature', col_wrap=5,
            kind='box', height=2.8, aspect=0.75, showfliers=False)

Swap 'x' and 'col' attributes to see the chart 'the other way around'

In [ ]:
g = sns.catplot(data=transformed_long_5_df, col='clusters_5', y='value', x='feature', col_wrap=5,
            kind='box', height=2.8, aspect=0.75, showfliers=False)
g.set_xticklabels(rotation=45, ha='right')

In [ ]:
g = sns.catplot(data=transformed_long_3_df, col='clusters_3', y='value', x='feature', col_wrap=5,
            kind='box', height=2.8, aspect=0.75, showfliers=False)
g.set_xticklabels(rotation=45, ha='right')

TODO: show symbols per cluster, requires symbol column in long format dataframe

### So what _is_ the right way to visualize these cluster?
Recall that we used PCA earlier but PCA has its limitations. It is not great for non-linear datasets. TSNE used to be the standard but the current standard is UMAP (unfortunately not part of scikit-learn)

In [ ]:
#!pip install umap-learn

Did you notice? You pip install `umap-learn`, not `umap`

In [ ]:
import umap

In [ ]:
%%time
embeddings = umap.UMAP().fit_transform(clean_df[MODEL_FEATURES])

umap_df = pd.DataFrame(embeddings, columns=['x', 'y'])
umap_df.head()

In [ ]:
pd.concat([umap_df, transformed_df.clusters_3], axis=1).plot.scatter('x', 'y', c='clusters_3', cmap='viridis')

In [ ]:
pd.concat([umap_df, transformed_df.clusters_5], axis=1).plot.scatter('x', 'y', c='clusters_5', cmap='viridis')